# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baraakhalid14/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
rule_description = """
A page should receive a higher action score when it already gets impressions,
has a useful ranking position, but receives fewer clicks than expected.
Older pages can also receive extra priority because their content may need refreshing.
"""

reason_codes = {
    "HIGH_IMPRESSIONS_LOW_CTR": "The page is visible in search but receives too few clicks.",
    "STRIKING_DISTANCE": "The page ranks close to the first page and may improve with optimization.",
    "STALE_CONTENT": "The page is old enough that its content may need refreshing.",
    "NO_ACTION": "The page does not currently meet the baseline action conditions."
}

print(rule_description)
print(reason_codes)


A page should receive a higher action score when it already gets impressions,
has a useful ranking position, but receives fewer clicks than expected.
Older pages can also receive extra priority because their content may need refreshing.

{'HIGH_IMPRESSIONS_LOW_CTR': 'The page is visible in search but receives too few clicks.', 'STRIKING_DISTANCE': 'The page ranks close to the first page and may improve with optimization.', 'STALE_CONTENT': 'The page is old enough that its content may need refreshing.', 'NO_ACTION': 'The page does not currently meet the baseline action conditions.'}


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
import pandas as pd

data_url = "https://raw.githubusercontent.com/baraakhalid14/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_url)

print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

df.head()

Rows and columns: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
keywords = [
    "impression",
    "click",
    "ctr",
    "position",
    "update",
    "age",
    "days",
    "date",
    "trend"
]

relevant_columns = [
    col for col in df.columns
    if any(word in col.lower() for word in keywords)
]

print("Relevant columns:")
for col in relevant_columns:
    print(col)

Relevant columns:
impressions_90d
clicks_90d
pageviews_90d
engaged_sessions_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
impressions_prev_30d
clicks_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
ctr
avg_position
engagement_rate
impression_tier
position_tier
trend_direction
trend_pct


In [11]:
import os
import numpy as np
import pandas as pd

# Make a working copy
ranked = df.copy()

# Convert the required columns to numbers
numeric_columns = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

for column in numeric_columns:
    ranked[column] = pd.to_numeric(ranked[column], errors="coerce").fillna(0)

# Transparent rule conditions
ranked["high_impressions"] = ranked["impressions_90d"] >= 500

ranked["low_ctr"] = (
    (ranked["impressions_90d"] >= 500) &
    (ranked["ctr"] < 1.0)
)

ranked["striking_distance"] = (
    (ranked["avg_position"] >= 4) &
    (ranked["avg_position"] <= 20)
)

ranked["stale_content"] = ranked["days_since_last_update"] >= 180

# Simple hand-written baseline score
ranked["action_score"] = (
    ranked["high_impressions"].astype(int) * 3 +
    ranked["low_ctr"].astype(int) * 4 +
    ranked["striking_distance"].astype(int) * 3 +
    ranked["stale_content"].astype(int) * 2
)

# Give each row one main reason code
conditions = [
    ranked["low_ctr"],
    ranked["striking_distance"],
    ranked["stale_content"]
]

reason_choices = [
    "HIGH_IMPRESSIONS_LOW_CTR",
    "STRIKING_DISTANCE",
    "STALE_CONTENT"
]

ranked["reason_code"] = np.select(
    conditions,
    reason_choices,
    default="NO_ACTION"
)

# Give each row one action label
action_choices = [
    "OPTIMIZE_TITLE_AND_SNIPPET",
    "IMPROVE_PAGE_FOR_HIGHER_RANKING",
    "REFRESH_CONTENT"
]

ranked["action_label"] = np.select(
    conditions,
    action_choices,
    default="NO_ACTION"
)

# Rank highest scores first
ranked = ranked.sort_values(
    by=["action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

ranked["rank"] = ranked.index + 1

# Select the columns required in the output
output_columns = [
    "rank",
    "content_id",
    "client_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "action_score",
    "reason_code",
    "action_label"
]

baseline_output = ranked[output_columns]

# Create the required folder and CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
baseline_output.to_csv(output_path, index=False)

print("CSV successfully created:", output_path)
print("Number of ranked rows:", len(baseline_output))

baseline_output.head(20)

CSV successfully created: work/outputs/baseline_action_score.csv
Number of ranked rows: 30000


,rank,content_id,client_id,impressions_90d,ctr,avg_position,days_since_last_update,action_score,reason_code,action_label
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,0.15,19.7,194,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
1,2,content_0a91db491d14,client_7f2253d7e2,13299,0.49,10.5,193,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
2,3,content_c2d929d83eaa,client_7f2253d7e2,7558,0.20,17.9,193,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
3,4,content_fe16a55cd13d,client_7f2253d7e2,4556,0.33,16.4,194,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
4,5,content_928af3e22c80,client_7f2253d7e2,1697,0.12,15.8,193,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
5,6,content_e3ff1b093148,client_d029fa3a95,1408,0.28,7.8,183,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
6,7,content_7f116ae1f6f5,client_9400f1b21c,954,0.42,9.0,301,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
7,8,content_77d4d5930e5e,client_7f2253d7e2,828,0.24,18.6,194,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
8,9,content_72496874f806,client_4ec9599fc2,821,0.24,5.8,301,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
9,10,content_6226ee6adc91,client_d029fa3a95,545,0.18,17.8,183,12,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
top20_review = baseline_output.head(20).copy()

top20_review["confidence_note"] = top20_review.apply(
    lambda row: (
        f"High priority because it has {row['impressions_90d']:.0f} impressions, "
        f"a CTR of {row['ctr']:.2f}, an average position of {row['avg_position']:.1f}, "
        f"and was last updated {row['days_since_last_update']:.0f} days ago."
    ),
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    lambda row: (
        "The recommendation may be wrong if the page has a naturally low CTR, "
        "if its search intent is already satisfied, if the data is seasonal, "
        "or if changing the title and snippet would not improve relevance."
    ),
    axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action_label",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review[review_columns]

,rank,content_id,action_label,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,High priority because it has 61678 impressions...,The recommendation may be wrong if the page ha...
1,2,content_0a91db491d14,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,High priority because it has 13299 impressions...,The recommendation may be wrong if the page ha...
2,3,content_c2d929d83eaa,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,"High priority because it has 7558 impressions,...",The recommendation may be wrong if the page ha...
3,4,content_fe16a55cd13d,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,"High priority because it has 4556 impressions,...",The recommendation may be wrong if the page ha...
4,5,content_928af3e22c80,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,"High priority because it has 1697 impressions,...",The recommendation may be wrong if the page ha...
5,6,content_e3ff1b093148,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,"High priority because it has 1408 impressions,...",The recommendation may be wrong if the page ha...
6,7,content_7f116ae1f6f5,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,"High priority because it has 954 impressions, ...",The recommendation may be wrong if the page ha...
7,8,content_77d4d5930e5e,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,"High priority because it has 828 impressions, ...",The recommendation may be wrong if the page ha...
8,9,content_72496874f806,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,"High priority because it has 821 impressions, ...",The recommendation may be wrong if the page ha...
9,10,content_6226ee6adc91,OPTIMIZE_TITLE_AND_SNIPPET,HIGH_IMPRESSIONS_LOW_CTR,"High priority because it has 545 impressions, ...",The recommendation may be wrong if the page ha...


In [13]:
final_output = baseline_output.merge(
    top20_review[
        [
            "content_id",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ],
    on="content_id",
    how="left"
)

final_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Final CSV updated successfully.")
print("Rows:", len(final_output))
print("Columns:", final_output.columns.tolist())

Final CSV updated successfully.
Rows: 30000
Columns: ['rank', 'content_id', 'client_id', 'impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'action_score', 'reason_code', 'action_label', 'confidence_note', 'what_would_make_it_wrong']


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [14]:
# Inspect weaker recommendations near the bottom of the top 20
weak_picks = ranked.head(20).sort_values(
    by=["action_score", "impressions_90d"],
    ascending=[True, True]
).head(5)

print("Five weaker picks from the top 20:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update",
            "action_score",
            "reason_code",
            "action_label"
        ]
    ]
)

print("""
Weak-pick review:
Some recommendations may be weaker when impressions are only slightly above the
threshold, when CTR differences are very small, or when page age does not really
mean the information is outdated.

Leakage check:
The score uses only current or historical input columns:
impressions_90d, ctr, avg_position, and days_since_last_update.
It does not use product-generated flags, future outcomes, future dates,
or label-derived columns. Therefore, no obvious feature leakage is present.
""")

Five weaker picks from the top 20:


,rank,content_id,impressions_90d,ctr,avg_position,days_since_last_update,action_score,reason_code,action_label
19,20,content_aa4baf490b43,256290,0.50,5.9,20,10,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
18,19,content_89e84d699e9e,275226,0.89,4.8,20,10,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
17,18,content_8e7ba84a972b,288426,0.92,4.8,20,10,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
16,17,content_36ff89c8214e,295097,0.05,7.3,104,10,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET
15,16,content_cb112fce36be,309910,0.16,5.6,104,10,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_TITLE_AND_SNIPPET



Weak-pick review:
Some recommendations may be weaker when impressions are only slightly above the
threshold, when CTR differences are very small, or when page age does not really
mean the information is outdated.

Leakage check:
The score uses only current or historical input columns:
impressions_90d, ctr, avg_position, and days_since_last_update.
It does not use product-generated flags, future outcomes, future dates,
or label-derived columns. Therefore, no obvious feature leakage is present.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.